In [62]:
!pip install streamlit
!pip install psycopg2-binary
!pip install bcrypt
!pip install pyjwt
!pip install python-dotenv
!pip install pyngrok

In [63]:
import streamlit as st
import psycopg2
import bcrypt
import jwt
import random
import smtplib
import pandas as pd
from email.message import EmailMessage

In [64]:
import psycopg2

conn = psycopg2.connect(
    host="ep-shy-moon-aoqwmibh-pooler.c-2.ap-southeast-1.aws.neon.tech",
    database="neondb",
    user="neondb_owner",
    password="npg_l67wGVxKCDHe",
    port="5432",
    sslmode="require"
)

cursor = conn.cursor()

print("✅ Database Connected Successfully!")

✅ Database Connected Successfully!


In [65]:
try:
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS users (
        id SERIAL PRIMARY KEY,
        name VARCHAR(100) NOT NULL,
        email VARCHAR(100) UNIQUE NOT NULL,
        password TEXT NOT NULL
    );
    """)

    conn.commit()

    print("✅ Users table created successfully!")

except Exception as e:
    print("❌ Error:", e)

✅ Users table created successfully!


In [66]:
 %%writefile app.py

import streamlit as st
import psycopg2
import bcrypt
import jwt
import smtplib
import random

from datetime import datetime, timedelta
from email.mime.text import MIMEText


# ==================================================
# PAGE SETTINGS
# ==================================================

st.set_page_config(
    page_title="Mood Mentor",
    page_icon="🌿",
    layout="wide",
    initial_sidebar_state="expanded"
)


# ==================================================
# MODERN LIGHT WELLNESS DESIGN
# ==================================================

st.markdown(
    """
    <style>

    .stApp {
        background:
        radial-gradient(
            circle at 10% 10%,
            #ddfff5 0%,
            transparent 30%
        ),
        radial-gradient(
            circle at 90% 10%,
            #ececff 0%,
            transparent 32%
        ),
        radial-gradient(
            circle at 80% 90%,
            #fff0e5 0%,
            transparent 30%
        ),
        linear-gradient(
            135deg,
            #fbfffe,
            #faf9ff,
            #fffaf6
        );

        color: #29465b;
    }


    .block-container {

        max-width: 1200px;

        padding-top: 2rem;

        padding-bottom: 3rem;

        animation:
        pageLoad
        0.7s
        ease-in-out;

    }


    @keyframes pageLoad {

        from {

            opacity: 0;

            transform:
            translateY(15px);

        }

        to {

            opacity: 1;

            transform:
            translateY(0);

        }

    }


    h1 {

        color: #315f68;

        font-weight: 850;

        letter-spacing: -1px;

    }


    h2,
    h3 {

        color: #4f766d;

        font-weight: 750;

    }


    p {

        color: #536b78;

        font-size: 17px;

        line-height: 1.7;

    }


    [data-testid="stSidebar"] {

        background:

        linear-gradient(
            180deg,
            #e2fff6,
            #edf0ff,
            #fff5ec
        );

        border-right:

        1px solid
        rgba(
            89,
            151,
            137,
            0.18
        );

    }


    [data-testid="stSidebar"] h1 {

        color: #315f68;

    }


    .stSelectbox > div > div {

        background:

        rgba(
            255,
            255,
            255,
            0.80
        );

        border-radius: 14px;

    }


    .stTextInput input {

        background:

        rgba(
            255,
            255,
            255,
            0.92
        );

        color: #243b53;

        border:

        1px solid
        #bfe4da;

        border-radius: 14px;

        padding: 12px;

        transition: 0.25s;

    }


    .stTextInput input:focus {

        border:

        2px solid
        #70cdb7;

        box-shadow:

        0 0 0 4px
        rgba(
            112,
            205,
            183,
            0.15
        );

    }


    .stButton > button {

        border: none;

        border-radius: 14px;

        padding:

        0.65rem
        1.7rem;

        color: white;

        font-weight: 750;

        background:

        linear-gradient(
            90deg,
            #65c9ae,
            #789ee4
        );

        box-shadow:

        0 8px 22px
        rgba(
            93,
            146,
            181,
            0.24
        );

        transition:

        transform
        0.25s,

        box-shadow
        0.25s;

    }


    .stButton > button:hover {

        transform:

        translateY(-3px);

        box-shadow:

        0 14px 28px
        rgba(
            93,
            146,
            181,
            0.32
        );

    }


    [data-testid="stAlert"] {

        border-radius: 17px;

        box-shadow:

        0 6px 18px
        rgba(
            78,
            113,
            126,
            0.08
        );

    }


    [data-testid="stFileUploader"] {

        background:

        rgba(
            255,
            255,
            255,
            0.75
        );

        border:

        2px dashed
        #91d7c7;

        border-radius: 22px;

        padding: 15px;

    }


    .hero-card {

        background:

        linear-gradient(
            135deg,
            rgba(
                255,
                255,
                255,
                0.94
            ),
            rgba(
                236,
                255,
                249,
                0.88
            ),
            rgba(
                242,
                241,
                255,
                0.88
            )
        );

        border:

        1px solid
        rgba(
            105,
            194,
            172,
            0.28
        );

        border-radius: 30px;

        padding:

        45px
        40px;

        box-shadow:

        0 18px 50px
        rgba(
            73,
            116,
            130,
            0.14
        );

        margin-bottom: 25px;

        transition: 0.3s;

    }


    .hero-card:hover {

        transform:

        translateY(-4px);

        box-shadow:

        0 24px 58px
        rgba(
            73,
            116,
            130,
            0.19
        );

    }


    .wellness-card {

        background:

        rgba(
            255,
            255,
            255,
            0.82
        );

        border:

        1px solid
        rgba(
            107,
            202,
            180,
            0.28
        );

        border-radius: 24px;

        padding: 25px;

        margin:

        14px
        0;

        box-shadow:

        0 12px 35px
        rgba(
            82,
            119,
            134,
            0.12
        );

        transition: 0.3s;

    }


    .wellness-card:hover {

        transform:

        translateY(-5px);

        box-shadow:

        0 18px 42px
        rgba(
            82,
            119,
            134,
            0.18
        );

    }


    .mood-banner {

        text-align: center;

        font-size: 45px;

        letter-spacing: 11px;

        padding:

        16px
        5px;

        animation:

        floatingMood
        3s
        ease-in-out
        infinite;

    }


    @keyframes floatingMood {

        0%,
        100% {

            transform:

            translateY(0);

        }

        50% {

            transform:

            translateY(-7px);

        }

    }


    .section-title {

        background:

        linear-gradient(
            90deg,
            #e6fff7,
            #eff0ff
        );

        border-left:

        6px solid
        #70cdb7;

        border-radius: 15px;

        padding:

        14px
        18px;

        margin:

        20px
        0;

        color: #315f68;

        font-size: 23px;

        font-weight: 800;

    }


    .metric-card {

        background:

        rgba(
            255,
            255,
            255,
            0.88
        );

        border-radius: 22px;

        padding: 23px;

        text-align: center;

        border:

        1px solid
        rgba(
            112,
            205,
            183,
            0.25
        );

        box-shadow:

        0 10px 28px
        rgba(
            73,
            116,
            130,
            0.11
        );

    }


    .metric-number {

        font-size: 34px;

        font-weight: 850;

        color: #527dc4;

    }


    .metric-label {

        font-size: 15px;

        color: #657c87;

    }


    .footer {

        text-align: center;

        color: #718096;

        padding-top: 40px;

        padding-bottom: 15px;

        font-size: 14px;

    }

    </style>
    """,

    unsafe_allow_html=True
)


# ==================================================
# DATABASE CONNECTION - NEON POSTGRESQL
# ==================================================

try:

    conn = psycopg2.connect(

        host="ep-shy-moon-aoqwmibh-pooler.c-2.ap-southeast-1.aws.neon.tech",

        database="neondb",

        user="neondb_owner",

        password="npg_l67wGVxKCDHe",

        port="5432",

        sslmode="require"

    )


    cursor = conn.cursor()


except Exception as error:


    st.error(

        "Database connection failed."

    )


    st.stop()


# ==================================================
# CREATE USERS TABLE
# ==================================================

try:


    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS users
        (
            id SERIAL PRIMARY KEY,

            name VARCHAR(100)
            NOT NULL,

            email VARCHAR(150)
            UNIQUE
            NOT NULL,

            password TEXT
            NOT NULL
        )
        """
    )


    conn.commit()


except Exception as error:


    conn.rollback()


    st.error(

        "Users table could not be created."

    )


# ==================================================
# SESSION MANAGEMENT
# ==================================================

if "logged_in" not in st.session_state:

    st.session_state[
        "logged_in"
    ] = False


if "username" not in st.session_state:

    st.session_state[
        "username"
    ] = ""


if "token" not in st.session_state:

    st.session_state[
        "token"
    ] = ""


if "otp" not in st.session_state:

    st.session_state[
        "otp"
    ] = None


if "reset_email" not in st.session_state:

    st.session_state[
        "reset_email"
    ] = ""


if "otp_verified" not in st.session_state:

    st.session_state[
        "otp_verified"
    ] = False


if "selected_mood" not in st.session_state:

    st.session_state[
        "selected_mood"
    ] = ""


# ==================================================
# SIDEBAR
# ==================================================

st.sidebar.markdown(
    """
    # 🌿 Mood Mentor

    **Employee Wellness Platform**

    Your well-being matters.
    """
)


st.sidebar.markdown("---")


menu = st.sidebar.selectbox(

    "🧭 Navigation",

    [

        "🏠 Home",

        "📝 Sign Up",

        "🔐 Login",

        "🔑 Forgot Password",

        "📊 Dashboard"

    ]

)


st.sidebar.markdown("---")


st.sidebar.info(
    """
    💚 Take a moment today
    to check in with yourself.
    """
)


# ==================================================
# HOME PAGE
# ==================================================

if menu == "🏠 Home":


    st.markdown(
    """
    <h1>🌿 Mood Mentor</h1>

    <h2>Employee Wellness Management Analytics</h2>

    <p>
    A secure and intelligent wellness platform designed to understand moods,
    encourage healthier habits and support employee well-being.
    </p>
    """,
    unsafe_allow_html=True
)


    st.markdown(
        """

        <div class="mood-banner">

        😭 😟 😕 😐 🙂 😊 🤩

        </div>

        """,

        unsafe_allow_html=True

    )


    st.markdown(
        """

        <div class="wellness-card">

        <h3>

        💚 Your feelings matter

        </h3>

        <p>

        Check in with your mood,
        understand your wellness patterns
        and take one positive step every day.

        </p>

        </div>

        """,

        unsafe_allow_html=True

    )


    col1, col2, col3 = st.columns(3)


    with col1:


        st.info(
            """
            🔐 **Secure Authentication**

            Password encryption,
            JWT authentication and
            secure sessions.
            """
        )


    with col2:


        st.success(
            """
            😊 **Seven Mood Levels**

            Express how you feel
            through an interactive
            mood check-in.
            """
        )


    with col3:


        st.warning(
            """
            📊 **Wellness Analytics**

            Upload employee wellness
            data for future insights.
            """
        )


# ==================================================
# SIGN-UP PAGE
# ==================================================

elif menu == "📝 Sign Up":


    st.markdown(
        """

        <div class="hero-card">

        <h1>

        📝 Create Your Account

        </h1>

        <p>

        Join Mood Mentor and begin
        your employee wellness journey.

        </p>

        </div>

        """,

        unsafe_allow_html=True

    )


    name = st.text_input(

        "👤 Full Name"

    )


    email = st.text_input(

        "📧 Email Address"

    )


    password = st.text_input(

        "🔒 Password",

        type="password"

    )


    confirm_password = st.text_input(

        "🔒 Confirm Password",

        type="password"

    )


    if password:


        password_score = min(

            len(password) / 10,

            1.0

        )


        st.progress(

            password_score

        )


        if len(password) < 6:


            st.caption(

                "🔴 Weak password"

            )


        elif len(password) < 10:


            st.caption(

                "🟡 Medium password"

            )


        else:


            st.caption(

                "🟢 Strong password"

            )


    if st.button(

        "✨ Create Account"

    ):


        if name.strip() == "":


            st.error(

                "Please enter your name."

            )


        elif email.strip() == "":


            st.error(

                "Please enter your email address."

            )


        elif "@" not in email:


            st.error(

                "Please enter a valid email address."

            )


        elif password == "":


            st.error(

                "Please enter your password."

            )


        elif len(password) < 6:


            st.error(

                "Password must contain at least 6 characters."

            )


        elif password != confirm_password:


            st.error(

                "Passwords do not match."

            )


        else:


            encrypted_password = bcrypt.hashpw(

                password.encode(

                    "utf-8"

                ),

                bcrypt.gensalt()

            ).decode(

                "utf-8"

            )


            try:


                cursor.execute(
                    """
                    INSERT INTO users
                    (
                        name,
                        email,
                        password
                    )

                    VALUES
                    (
                        %s,
                        %s,
                        %s
                    )
                    """,

                    (

                        name.strip(),

                        email.strip().lower(),

                        encrypted_password

                    )

                )


                conn.commit()


                st.success(

                    "🎉 Registration Successful"

                )


                st.balloons()


                st.info(

                    "Open Login from the sidebar."

                )


            except psycopg2.errors.UniqueViolation:


                conn.rollback()


                st.error(

                    "This email address is already registered."

                )


            except Exception as error:


                conn.rollback()


                st.error(

                    "Registration failed. Please try again."

                )


# ==================================================
# LOGIN PAGE
# ==================================================

elif menu == "🔐 Login":


    st.markdown(
        """

        <div class="hero-card">

        <h1>

        🔐 Welcome Back

        </h1>

        <p>

        Log in securely to continue
        your wellness journey.

        </p>

        </div>

        """,

        unsafe_allow_html=True

    )


    email = st.text_input(

        "📧 Email Address"

    )


    password = st.text_input(

        "🔒 Password",

        type="password"

    )


    if st.button(

        "🚀 Login Securely"

    ):


        if email.strip() == "":


            st.error(

                "Please enter your email address."

            )


        elif password == "":


            st.error(

                "Please enter your password."

            )


        else:


            try:


                cursor.execute(
                    """
                    SELECT

                    name,

                    password

                    FROM users

                    WHERE email = %s
                    """,

                    (

                        email.strip().lower(),

                    )

                )


                user = cursor.fetchone()


                if user is None:


                    st.error(

                        "User not found."

                    )


                else:


                    name = user[0]


                    stored_password = user[1]


                    password_correct = bcrypt.checkpw(

                        password.encode(

                            "utf-8"

                        ),

                        stored_password.encode(

                            "utf-8"

                        )

                    )


                    if password_correct:


                        token = jwt.encode(
                            {

                                "email":

                                email.strip().lower(),


                                "exp":

                                datetime.utcnow()

                                +

                                timedelta(

                                    hours=1

                                )

                            },

                            "MoodMentorSecretKey",

                            algorithm="HS256"

                        )


                        st.session_state[

                            "logged_in"

                        ] = True


                        st.session_state[

                            "username"

                        ] = name


                        st.session_state[

                            "token"

                        ] = token


                        st.success(

                            "✅ Login Successful"

                        )


                        st.balloons()


                        st.info(

                            "Open Dashboard from the sidebar."

                        )


                    else:


                        st.error(

                            "Incorrect Password"

                        )


            except Exception as error:


                st.error(

                    "Login failed. Please try again."

                )


# ==================================================
# FORGOT PASSWORD PAGE
# ==================================================

elif menu == "🔑 Forgot Password":


    st.markdown(
        """

        <div class="hero-card">

        <h1>

        🔑 Recover Your Account

        </h1>

        <p>

        Verify your registered email
        using a secure one-time password.

        </p>

        </div>

        """,

        unsafe_allow_html=True

    )


    reset_email = st.text_input(

        "📧 Registered Email Address"

    )


    if st.button(

        "📨 Send OTP"

    ):


        if reset_email.strip() == "":


            st.error(

                "Please enter your registered email address."

            )


        else:


            try:


                cursor.execute(
                    """
                    SELECT email

                    FROM users

                    WHERE email = %s
                    """,

                    (

                        reset_email.strip().lower(),

                    )

                )


                registered_user = cursor.fetchone()


                if registered_user is None:


                    st.error(

                        "This email address is not registered."

                    )


                else:


                    otp = random.randint(

                        100000,

                        999999

                    )


                    sender_email = (

                        "somaanil14@gmail.com"

                    )


                    google_app_password = (

                        "anon kaqe hfws peli"

                    )


                    message = MIMEText(
                        f"""

Hello,

Your Mood Mentor password-reset OTP is:

{otp}

Do not share this OTP with anyone.

Mood Mentor
Employee Wellness Management Analytics

                        """

                    )


                    message[

                        "Subject"

                    ] = (

                        "Mood Mentor Password Reset OTP"

                    )


                    message[

                        "From"

                    ] = sender_email


                    message[

                        "To"

                    ] = (

                        reset_email

                        .strip()

                        .lower()

                    )


                    smtp_server = smtplib.SMTP(

                        "smtp.gmail.com",

                        587

                    )


                    smtp_server.starttls()


                    smtp_server.login(

                        sender_email,

                        google_app_password

                    )


                    smtp_server.send_message(

                        message

                    )


                    smtp_server.quit()


                    st.session_state[

                        "otp"

                    ] = str(

                        otp

                    )


                    st.session_state[

                        "reset_email"

                    ] = (

                        reset_email

                        .strip()

                        .lower()

                    )


                    st.session_state[

                        "otp_verified"

                    ] = False


                    st.success(

                        "📨 OTP sent successfully. Check your Gmail inbox."

                    )


            except Exception as error:


                st.error(

                    """
                    OTP could not be sent.
                    Check your Gmail address
                    and Google App Password.
                    """

                )


    entered_otp = st.text_input(

        "🔢 Enter Six-Digit OTP"

    )


    if st.button(

        "✅ Verify OTP"

    ):


        if st.session_state[

            "otp"

        ] is None:


            st.error(

                "First enter your email and click Send OTP."

            )


        elif entered_otp.strip() == str(

            st.session_state[

                "otp"

            ]

        ):


            st.session_state[

                "otp_verified"

            ] = True


            st.success(

                "OTP verified successfully. Create your new password."

            )


        else:


            st.error(

                "Invalid OTP."

            )


    if st.session_state[

        "otp_verified"

    ]:


        new_password = st.text_input(

            "🔒 New Password",

            type="password"

        )


        confirm_new_password = st.text_input(

            "🔒 Confirm New Password",

            type="password"

        )


        if st.button(

            "🔄 Reset Password"

        ):


            if len(

                new_password

            ) < 6:


                st.error(

                    "Password must contain at least 6 characters."

                )


            elif new_password != confirm_new_password:


                st.error(

                    "New passwords do not match."

                )


            else:


                try:


                    encrypted_new_password = bcrypt.hashpw(

                        new_password.encode(

                            "utf-8"

                        ),

                        bcrypt.gensalt()

                    ).decode(

                        "utf-8"

                    )


                    cursor.execute(
                        """
                        UPDATE users

                        SET password = %s

                        WHERE email = %s
                        """,

                        (

                            encrypted_new_password,

                            st.session_state[

                                "reset_email"

                            ]

                        )

                    )


                    conn.commit()


                    st.success(

                        "🎉 Password reset successfully. You can now log in."

                    )


                    st.balloons()


                    st.session_state[

                        "otp"

                    ] = None


                    st.session_state[

                        "reset_email"

                    ] = ""


                    st.session_state[

                        "otp_verified"

                    ] = False


                except Exception as error:


                    conn.rollback()


                    st.error(

                        "Password could not be updated."

                    )


# ==================================================
# DASHBOARD PAGE
# ==================================================

elif menu == "📊 Dashboard":


    if st.session_state[

        "logged_in"

    ]:


        st.markdown(
            f"""

            <div class="hero-card">

            <h1>

            🌿 Welcome,
            {st.session_state["username"]}!

            </h1>

            <p>

            This is your personal
            employee wellness space.

            Take a moment to understand
            how you feel today.

            </p>

            </div>

            """,

            unsafe_allow_html=True

        )


        st.markdown(
            """

            <div class="section-title">

            😊 How are you feeling today?

            </div>

            """,

            unsafe_allow_html=True

        )


        mood_options = {

            "😭":

            (
                "Very Low",

                "You may be having a difficult day. "
                "Consider talking with someone you trust "
                "and taking time to rest.",

                15

            ),


            "😟":

            (
                "Low",

                "Be gentle with yourself today. "
                "A short break or a calm conversation "
                "may help.",

                30

            ),


            "😕":

            (
                "Not Good",

                "Your feelings deserve attention. "
                "Try one small activity that brings "
                "you comfort.",

                45

            ),


            "😐":

            (
                "Neutral",

                "A balanced day is a good opportunity "
                "to check your needs and recharge.",

                55

            ),


            "🙂":

            (
                "Good",

                "You are doing well. "
                "Keep supporting your positive habits.",

                70

            ),


            "😊":

            (
                "Very Good",

                "Wonderful! Carry this positive energy "
                "into your work and relationships.",

                85

            ),


            "🤩":

            (
                "Excellent",

                "Amazing! Celebrate your positive mood "
                "and share encouragement with others.",

                100

            )

        }


        selected_mood = st.radio(

            "Select one mood",

            list(

                mood_options.keys()

            ),

            horizontal=True,

            label_visibility="collapsed"

        )


        mood_name = mood_options[

            selected_mood

        ][0]


        mood_message = mood_options[

            selected_mood

        ][1]


        wellness_score = mood_options[

            selected_mood

        ][2]


        st.session_state[

            "selected_mood"

        ] = selected_mood


        st.markdown(
            f"""

            <div class="wellness-card">

            <h2>

            {selected_mood}
            {mood_name}

            </h2>

            <p>

            {mood_message}

            </p>

            </div>

            """,

            unsafe_allow_html=True

        )


        st.write(

            f"**Today's Wellness Score: "
            f"{wellness_score}/100**"

        )


        st.progress(

            wellness_score

            /

            100

        )


        st.markdown(
            """

            <div class="section-title">

            🌱 Today's Wellness Suggestions

            </div>

            """,

            unsafe_allow_html=True

        )


        tip1, tip2, tip3 = st.columns(3)


        with tip1:


            st.info(
                """
                💧 **Hydrate**

                Drink water and take
                a short screen break.
                """
            )


        with tip2:


            st.success(
                """
                🧘 **Breathe**

                Take five slow,
                mindful breaths.
                """
            )


        with tip3:


            st.warning(
                """
                🚶 **Move**

                Walk or stretch
                for five minutes.
                """
            )


        st.markdown(
            """

            <div class="section-title">

            📊 Employee Wellness Data

            </div>

            """,

            unsafe_allow_html=True

        )


        uploaded_file = st.file_uploader(

            "Upload Employee Wellness CSV File",

            type=[

                "csv"

            ]

        )


        if uploaded_file is not None:


            st.success(

                "CSV file uploaded successfully."

            )


            st.info(

                """
                The dataset is ready
                for wellness analytics
                in the next milestone.
                """

            )


        st.markdown(
            """

            <div class="section-title">

            ✨ Daily Wellness Reminder

            </div>

            """,

            unsafe_allow_html=True

        )


        st.markdown(
            """

            <div class="wellness-card">

            <h3>

            “Small positive actions,
            repeated every day,
            create meaningful change.”

            </h3>

            </div>

            """,

            unsafe_allow_html=True

        )


        if st.button(

            "🚪 Logout"

        ):


            st.session_state[

                "logged_in"

            ] = False


            st.session_state[

                "username"

            ] = ""


            st.session_state[

                "token"

            ] = ""


            st.session_state[

                "selected_mood"

            ] = ""


            st.success(

                "Logout Successful"

            )


            st.rerun()


    else:


        st.warning(

            "🔐 Please log in before opening the Dashboard."

        )


        st.info(

            "Open Login from the sidebar to continue."

        )


# ==================================================
# FOOTER
# ==================================================

st.markdown(
    """

    <div class="footer">

    🌿 Mood Mentor

    <br>

    Employee Wellness Management Analytics

    <br>

    Secure • Supportive • Insightful

    </div>

    """,

    unsafe_allow_html=True

)

Overwriting app.py


In [67]:
from pyngrok import ngrok

ngrok.set_auth_token("3GEQ8y2fzDDKwnq1AQeAOii4Gfy_82omE4sB635nPiDsCq95W")

In [68]:
from pyngrok import ngrok

# Kill any running ngrok processes to ensure a clean state
ngrok.kill()

# Set the correct authtoken explicitly before connecting
ngrok.set_auth_token("3GEVyqlIdEPSwaYUVGoKr97ZKQX_4pU1V84u6GdxFBAHSX6wM")

public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://pamperer-twilight-paper.ngrok-free.dev" -> "http://localhost:8501"
